# 03 · 七模型 embedding：重复性、扰动信号与批次影响

对 **MAE、CA-MAE、DiffMAE、CNN-MAE、OpenPhenom、MAE-L/8、MAE-G/8** 完成同一套诊断。
主问题是：①相同扰动是否比非重复更相似；②扰动相对对照的变化是否稳定；③experiment/plate 是否形成结构；④扰动信号能否跨越批次。

分析单位是单个 well，七模型按 well_id 取交集，剔除任一模型的无效/零向量后使用同一套孔位、孔对和抽样。
药物重复＝化合物＋浓度；CRISPR 重复＝同 guide；同基因其他 guide 不当作严格非重复。
plate 唯一键为 experiment/plate，阴性对照按显式规则匹配。这里使用**有符号余弦相似度**。

这是描述性诊断：默认分析已提供的 train＋val，保留 split 标记；包含训练孔的图不代表留出泛化成绩。
七模型的上游数据和预算不同，公开模型的预训练重叠也未审计。原 val-query/train-gallery 结果保留在附录。
主分析不执行 PCA-CS/按板中心化；降维仅用于画图，定量指标始终在原 embedding 的余弦空间计算。

In [ ]:
import importlib.util
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(os.environ['PHENOSTATE_PROJECT_ROOT']).expanduser() if os.environ.get('PHENOSTATE_PROJECT_ROOT') else next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file()
)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from phenostate.diagnostic_inputs import load_diagnostic_inputs
from phenostate.representation_diagnostics import (
    build_pair_plan, score_pair_plan, summarize_effects, control_neighbor_enrichment, validate_inputs,
)
from phenostate.diagnostic_plots import (
    plot_pair_distributions, plot_effects, plot_effect_summary,
    make_projection_coordinates, plot_projections, plot_similarity_heatmaps, plot_neighbor_summary,
)

MODELS = ['mae', 'ca_mae', 'diffmae', 'cnn_mae', 'OpenPhenom', 'MAE-L-8', 'MAE-G-8']
PUBLIC_MODELS = {'OpenPhenom', 'MAE-L-8', 'MAE-G-8'}
RESULT_DIR = Path(os.environ.get('PHENOSTATE_RESULTS_DIR', PROJECT_ROOT / 'results/downstream_val_all')).expanduser()
DATA_ROOT = Path(os.environ.get('RXRX3_CORE_ROOT', '/home/xkj2006/CellPainting/rxrx3_core')).expanduser()
EMBEDDING_ROOT = Path(os.environ.get('PHENOSTATE_EMBEDDING_ROOT', PROJECT_ROOT.parent / 'data/embeddings')).expanduser()
PUBLIC_ROOT = Path(os.environ.get('PHENOSTATE_PUBLIC_EMBEDDING_ROOT', DATA_ROOT)).expanduser()
METADATA_PATH = Path(os.environ.get('PHENOSTATE_METADATA_PATH', DATA_ROOT / 'metadata_rxrx3_core.csv')).expanduser()
OUTPUT_DIR = Path(os.environ.get('PHENOSTATE_DIAGNOSTIC_OUTPUT', RESULT_DIR / 'embedding_diagnostics_7_models')).expanduser()

# 这里可直接修改七个模型路径。默认优先复用 RESULT_DIR/protocol.json 中有效的来源路径。
MODEL_FILES = {
    name: [EMBEDDING_ROOT / folder / f'embedding_{split}.parquet' for split in ('train', 'val')]
    for name, folder in [('mae', 'mae_patch8_head8'), ('ca_mae', 'ca_mae'), ('diffmae', 'diffmae'), ('cnn_mae', 'cnn_mae')]
}
MODEL_FILES.update({
    'OpenPhenom': [PUBLIC_ROOT / 'OpenPhenom_rxrx3_core_embeddings.parquet'],
    'MAE-L-8': [PUBLIC_ROOT / 'MAE-L_8_RPI-93M_rxrx3_core_embeddings.parquet'],
    'MAE-G-8': [PUBLIC_ROOT / 'MAE-G_8_PP-16M_trimmed_rxrx3_core_embeddings.parquet'],
})
SEED = 2026
SELECTED_SPLITS = ('train', 'val')  # 可改为 ('val',)；不自动纳入 test，不生成新 split。
MAX_PERTURBATIONS_PER_KIND = 200  # 优先跨批次覆盖；不按某模型得分/活性筛选。
MAX_WELLS_PER_PERTURBATION_PLATE = 3
MAX_CONTROLS_PER_PLATE = 32
MAX_ANCHORS_PER_PERTURBATION = 8
MAX_REFERENCE_PLATES = 4
MAX_PAIRS_PER_RELATION = 24
NEIGHBOR_K = 10
N_BOOTSTRAP = 500
MAX_PROJECTION_POINTS = 1500
PROJECTION_METHODS = ('PCA', 'UMAP') if importlib.util.find_spec('umap') else ('PCA',)
SAVE_FIGURES = True

protocol_path = RESULT_DIR / 'protocol.json'
source_protocol = json.loads(protocol_path.read_text(encoding='utf-8')) if protocol_path.is_file() else {}
aliases = {
    'mae': ('mae', 'mae_patch8_head8', 'local_MAE'), 'ca_mae': ('ca_mae',),
    'diffmae': ('diffmae',), 'cnn_mae': ('cnn_mae',), 'OpenPhenom': ('OpenPhenom',),
    'MAE-L-8': ('MAE-L-8', 'Phenom-1'), 'MAE-G-8': ('MAE-G-8', 'Phenom-2'),
}
for name in MODELS:
    choices = [source_protocol.get('models', {}).get(alias) for alias in aliases[name]]
    choices = [entry for entry in choices if entry]
    if len(choices) > 1:
        raise ValueError(f'{name}: multiple protocol aliases; set MODEL_FILES explicitly.')
    if choices:
        paths = [Path(entry['path']).expanduser() for entry in choices[0].get('files', [])]
        if paths and all(p.is_file() for p in paths):
            MODEL_FILES[name] = paths

# 显式 JSON 也可覆盖所有模型路径，适合迁移服务器目录或自动化运行。
if os.environ.get('PHENOSTATE_DIAGNOSTIC_INPUTS'):
    config_path = Path(os.environ['PHENOSTATE_DIAGNOSTIC_INPUTS']).expanduser()
    configured = json.loads(config_path.read_text(encoding='utf-8'))
    MODEL_FILES = {name: [Path(p).expanduser() for p in files] for name, files in configured.items()}
MODEL_FILES = {name: [Path(p).expanduser() for p in files] for name, files in MODEL_FILES.items()}
if set(MODEL_FILES) != set(MODELS):
    raise ValueError(f'Exactly seven models required. Missing: {set(MODELS)-set(MODEL_FILES)}; extra: {set(MODEL_FILES)-set(MODELS)}')

split_candidates = []
if os.environ.get('PHENOSTATE_SPLITS_PATH'):
    split_candidates.append(Path(os.environ['PHENOSTATE_SPLITS_PATH']).expanduser())
elif os.environ.get('RXRX3_MDS_ROOT'):
    split_candidates.append(Path(os.environ['RXRX3_MDS_ROOT']).expanduser() / 'plate_splits.json')
elif source_protocol.get('split_source') not in (None, 'metadata split column'):
    p = Path(source_protocol['split_source']).expanduser()
    split_candidates.append(p / 'plate_splits.json' if p.is_dir() else p)
if not split_candidates:
    split_candidates.append(RESULT_DIR / 'well_manifest.csv')
SPLITS_PATH = next((p for p in split_candidates if p.is_file()), None)
if split_candidates and SPLITS_PATH is None and (os.environ.get('PHENOSTATE_SPLITS_PATH') or os.environ.get('RXRX3_MDS_ROOT')):
    raise FileNotFoundError(f'Configured saved split not found: {split_candidates}')

CONTROL_RULES_PATH = Path(os.environ.get('PHENOSTATE_CONTROL_RULES', PROJECT_ROOT / 'configs/negative_controls.example.json')).expanduser()
CONTROL_RULES = (json.loads(CONTROL_RULES_PATH.read_text(encoding='utf-8'))
                 if os.environ.get('PHENOSTATE_CONTROL_RULES') or not source_protocol.get('control_rules')
                 else source_protocol['control_rules'])
file_inventory = pd.DataFrame([{'model': name, 'path': str(p), 'present': p.is_file()}
                               for name in MODELS for p in MODEL_FILES[name]])
display(file_inventory)
display(pd.Series({'metadata': str(METADATA_PATH), 'splits': str(SPLITS_PATH), 'selected_splits': SELECTED_SPLITS,
                   'output': str(OUTPUT_DIR), 'projection_methods': PROJECTION_METHODS}, name='setting').to_frame())
if not file_inventory.present.all() or not METADATA_PATH.is_file():
    raise FileNotFoundError('Seven-model diagnostics require every input above plus metadata. Set the path variables/environment before running; no model is silently omitted.')
print('Control rules (verify actual negative-control semantics):', json.dumps(CONTROL_RULES, ensure_ascii=False))

## 1. 七模型共同孔位与覆盖率

读取七种 Parquet 表征，不需要 GPU/checkpoint/原始图像。先扫描共同 well_id，再按元数据固定抽样；每类默认最多 200 个扰动、每扰动每板最多 3 个孔、每板每类匹配对照最多 32 个。
抽样优先覆盖多个 plate/experiment，平局用固定 seed；这是有记录的覆盖子集，不是全库无偏分数。
任一模型的 NaN/Inf/零向量在七模型中共同剔除，并显示剔除原因。模型特征维数可以不同。

**严格标签与匹配条件**：同 guide 才是 CRISPR 重复；药物固定化合物＋规范化浓度；同基因其他 guide 排除出非重复。
不同 cell_type 或额外 match_fields 分开计算，图中的基本汇总单位为扰动×匹配 context。
缺对照/重复不补零，不把 EMPTY_control 自动解释为 DMSO。

In [ ]:
inputs = load_diagnostic_inputs(
    {name: MODEL_FILES[name] for name in MODELS}, METADATA_PATH,
    splits_path=SPLITS_PATH, control_rules=CONTROL_RULES, selected_splits=SELECTED_SPLITS, seed=SEED,
    max_perturbations_per_kind=MAX_PERTURBATIONS_PER_KIND,
    max_wells_per_perturbation_plate=MAX_WELLS_PER_PERTURBATION_PLATE,
    max_controls_per_plate=MAX_CONTROLS_PER_PLATE,
)
meta, embeddings = inputs['meta'], inputs['embeddings']
embeddings = validate_inputs(meta, embeddings)  # 七模型统一为单位向量；不拟合批次校正。
assert list(embeddings) == MODELS and all(len(x) == len(meta) for x in embeddings.values())
display(inputs['coverage'])
display(pd.Series(inputs['report'], name='value').to_frame())
display(meta.groupby(['kind', 'split', 'is_control'], dropna=False).agg(
    wells=('well_id', 'size'), experiments=('experiment_name', 'nunique'), plates=('batch_id', 'nunique')))
display(inputs['exclusions'].head(30))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
meta.to_csv(OUTPUT_DIR / 'well_manifest.csv', index=False)
inputs['coverage'].to_csv(OUTPUT_DIR / 'input_coverage.csv', index=False)
inputs['exclusions'].to_csv(OUTPUT_DIR / 'input_exclusions.csv', index=False)

def show_figure(fig, filename):
    if SAVE_FIGURES:
        fig.savefig(OUTPUT_DIR / f'{filename}.png', dpi=150, bbox_inches='tight')
    display(fig)
    plt.close(fig)

treated = meta.loc[~meta.is_control]
coverage_table = pd.crosstab(treated.perturbation, treated.batch_id)
coverage_table.to_csv(OUTPUT_DIR / 'perturbation_plate_coverage.csv')
coverage_rank = coverage_table.gt(0).sum(axis=1).sort_values(ascending=False, kind='stable')
display(coverage_rank.rename('n_plates').to_frame().head(20))
if not coverage_table.empty:
    visible = coverage_table.loc[coverage_rank.head(32).index]
    fig, ax = plt.subplots(figsize=(max(10, min(22, visible.shape[1] * 0.22)), 9))
    im = ax.imshow(visible, aspect='auto', interpolation='nearest', cmap='Blues')
    ax.set_yticks(range(len(visible)), visible.index, fontsize=6)
    step = max(1, visible.shape[1] // 60)
    ax.set_xticks(range(0, visible.shape[1], step), visible.columns[::step], rotation=90, fontsize=6)
    ax.set(title='Coverage: highest-coverage perturbations (metadata selection)', xlabel='Experiment / plate', ylabel='Strict perturbation')
    fig.colorbar(im, ax=ax, label='Selected wells')
    fig.tight_layout()
    show_figure(fig, '01_coverage')

## 2. 相似度：重复、非重复、扰动与阴性对照

七模型使用同一份 pair plan。每个扰动最多 8 个 anchor，尽量覆盖不同 plate；每 anchor 最多参考 4 个跨板重复板，每种关系最多 24 次比较。

| 图中关系 | 比较条件 | 用途 |
|---|---|---|
| Repeat / cross plate | 同扰动、同 experiment、不同 plate | 严格跨板重复性 |
| Nonrepeat / cross plate | 同 anchor、相同候选板的合格非重复 | 重复优势的参照 |
| Nonrepeat / same plate | 同 anchor、与跨板参照共享非重复身份的同板孔 | 比较批次与扰动 |
| Treated / control | 同板、同 kind、同 context、匹配阴性对照 | 相对正常状态的变化 |
| Control / same plate | 同类对照，不包含自身 | 正常波动基线 |
| Control / cross plate or experiment | 同类对照、同 context，跨批次 | 批次相关偏移 |

同板有严格重复时也展示。跨 experiment 仅比较确切相同的 guide 或化合物＋剂量；没有确切重复就记为缺失，不把同基因其他 guide 混入。
先对 anchor/候选板块取相似度中位数，再在扰动×context 内平均；图中每个点对应一个组，不把相关孔对当成独立样本。

In [ ]:
pair_plan, pair_exclusions = build_pair_plan(
    meta, seed=SEED, max_anchors_per_perturbation=MAX_ANCHORS_PER_PERTURBATION,
    max_reference_plates=MAX_REFERENCE_PLATES, max_pairs_per_relation=MAX_PAIRS_PER_RELATION,
)
display(pair_plan.groupby(['kind', 'relation']).agg(comparisons=('left', 'size'), blocks=('block', 'nunique')))
display(pair_exclusions.groupby(['kind', 'reason']).size().rename('anchor_count').to_frame() if not pair_exclusions.empty else pair_exclusions)
diagnostics = score_pair_plan(embeddings, meta, pair_plan)
pair_groups, effects = diagnostics['pair_groups'], diagnostics['effects']
pair_plan.to_csv(OUTPUT_DIR / 'pair_plan.csv', index=False)
pair_exclusions.to_csv(OUTPUT_DIR / 'pair_exclusions.csv', index=False)
for name, frame in diagnostics.items():
    frame.to_csv(OUTPUT_DIR / f'{name}.csv', index=False)
show_figure(plot_pair_distributions(pair_groups, MODELS, zoom=True), '02_similarity_distributions')
# 主图使用跨七模型统一的实际数值范围放大差异；完整 [-1,1] 色轴版本同时保存。
full_scale = plot_pair_distributions(pair_groups, MODELS, zoom=False)
if SAVE_FIGURES:
    full_scale.savefig(OUTPUT_DIR / '02_similarity_distributions_full_scale.png', dpi=150, bbox_inches='tight')
plt.close(full_scale)

## 3. 表型变化是否明显，而且可重复？

横轴为对照校准的表型偏离：`median(cos(control,control')) − median(cos(treated,control))`，即 `d(T,C)−d(C,C')`，其中 `d=1−cos`。
两项在同板匹配的控制块计算后，先对扰动×context 汇总。纵轴为跨板重复相似度减匹配非重复相似度。

| 位置 | 解读 |
|---|---|
| 右上 | 明显且可重复的表型变化 |
| 左上 | 变化较弱，但重复状态稳定 |
| 右下 | 变化大而不稳定，需要检查异常或条件差异 |
| 左下 | 当前表征中可辨识的扰动信号较弱 |

弱扰动接近对照是合理现象；横轴不是模型越大越好的成绩。零线是参照，不是显著性阈值。
缺跨板重复或同板对照的组保留在明细表，该散点仅展示两个指标均可计算的组，并报告实际数量。

In [ ]:
display(effects.groupby(['model', 'kind'])[['replicate_advantage', 'control_deviation']].count())
show_figure(plot_effects(effects, MODELS), '03_strength_vs_reproducibility')

## 4. experiment / plate 是否形成聚集？

使用三套**七模型共同、元数据平衡抽样**的 wells：全部选定孔位、同类阴性对照、最多 12 个跨批次覆盖充分的严格扰动。
每个模型、每个子集只计算一次降维坐标，随后分别按 experiment、plate、kind 或 perturbation 着色；坐标与所用 well_id 保存。

PCA 始终提供；若环境安装 `umap-learn`，追加 UMAP。UMAP 仅供观察，定量比较不使用二维距离。
PCA/UMAP 在所示探索子集拟合，不是性能评估的训练变换。不同模型独立坐标的位置和簇间距离不直接比较。
对照图中若含多个生物 context，请结合 context 明细解释，不能把真实细胞条件分离全部归因于技术批次。

全体样本按 experiment 分群也可能由扰动组成差异造成，因此同时查看同类对照与共享扰动，以及后面的条件化近邻指标。

In [ ]:
projection_reports = []
for method in PROJECTION_METHODS:
    for subset in ('all', 'controls', 'shared'):
        coordinates, projection_info = make_projection_coordinates(
            embeddings, meta, MODELS, subset=subset, method=method,
            max_points=MAX_PROJECTION_POINTS, max_perturbations=12, seed=SEED,
        )
        projection_reports.append({'method': method, 'subset': subset, **projection_info})
        if coordinates.empty:
            print(method, subset, 'NA:', projection_info)
            continue
        coordinates.to_csv(OUTPUT_DIR / f'coordinates_{method.lower()}_{subset}.csv', index=False)
        color_fields = ('experiment_name', 'batch_id', 'perturbation') if subset == 'shared' else (
            ('experiment_name', 'batch_id') if subset == 'controls' else ('experiment_name', 'batch_id', 'kind'))
        for color_by in color_fields:
            show_figure(plot_projections(coordinates, MODELS, color_by=color_by),
                        f'04_{method.lower()}_{subset}_{color_by}')
# 额外展示每个 kind 的一个 experiment 内的 plate 结构，避免全局 plate 颜色过多。
for kind in sorted(meta.kind.unique()):
    experiment_coverage = meta.loc[meta.is_control & meta.kind.eq(kind)].groupby('experiment_name').agg(
        plates=('batch_id', 'nunique'), wells=('well_id', 'size')).sort_values(['plates', 'wells'], ascending=False, kind='stable')
    if experiment_coverage.empty:
        continue
    experiment = experiment_coverage.index[0]  # 可在这里指定另一个 experiment；按覆盖选，不按模型效果选。
    focus_mask = (meta.kind.eq(kind) & meta.experiment_name.eq(experiment)).to_numpy()
    focus_meta = meta.loc[focus_mask].reset_index(drop=True)
    focus_embeddings = {name: values[focus_mask] for name, values in embeddings.items()}
    focus_coordinates, focus_info = make_projection_coordinates(
        focus_embeddings, focus_meta, MODELS, subset='controls', method='PCA', max_points=MAX_PROJECTION_POINTS, seed=SEED)
    projection_reports.append({**focus_info, 'subset': 'within_experiment_controls', 'experiment': experiment, 'kind': kind})
    if not focus_coordinates.empty:
        print('Within-experiment plate view:', kind, experiment)
        focus_coordinates.to_csv(OUTPUT_DIR / f'coordinates_pca_controls_{experiment}.csv', index=False)
        show_figure(plot_projections(focus_coordinates, MODELS, color_by='batch_id'), f'04_pca_controls_within_{experiment}')
display(pd.DataFrame(projection_reports))
if 'UMAP' not in PROJECTION_METHODS:
    print('Optional UMAP is unavailable; all seven models still receive PCA and high-dimensional quantitative diagnostics.')

## 5. 相似度热图：按批次排序 vs 按扰动排序

对固定的小组 wells 计算有符号余弦矩阵，七模型使用相同的元数据抽样与色标 `[-1,1]`。
同一个矩阵分别按 experiment/plate 和严格扰动排序；隐藏自身相似度的对角线。
批次排序出现高相似度方块提示批次相关结构；扰动排序出现跨批次高相似度支持重复信号。
热图是有限样本的直观诊断，不计算全库平方矩阵。

In [ ]:
for model in MODELS:
    show_figure(plot_similarity_heatmaps(embeddings, meta, model, max_points=160, seed=SEED),
                f'05_similarity_heatmap_{model}')

## 6. 扰动信号与 plate 信号的直接比较

令 B＝同扰动跨板相似度，C＝异扰动同板相似度，D＝异扰动跨板相似度。
对每个 anchor 与参考板，C 和 D 使用两板共享、相同剂量/合格基因的非重复身份集合，然后才计算差值。

| 指标 | 定义 | 如何解释 |
|---|---|---|
| Replicate advantage | B − D | 跨板后，相同扰动保留多少相似度优势；通常希望为正 |
| Competition | B − C | 同扰动跨板是否比不同扰动同板更相似；通常希望为正 |
| Plate advantage | C − D | 固定非重复身份后，同板带来多少相似度增加；正值提示板相关结构 |
| Experiment replicate advantage | 同扰动跨 experiment − 匹配非重复 | 只在确切扰动身份跨 experiment 存在时评价 |

若两板缺少共享非重复身份，仍可计算有合格参照的重复优势；Competition/Plate advantage 明确记 NA。
同板没有严格重复时不补造四格左上角。同一个差值可受弱表型、噪声和实验条件影响，不单独宣称因果或百分比方差归属。
置信区间按基因/化合物聚类 bootstrap，保留同基因多个 guide、同化合物多个浓度的相关性；不是训练随机种子误差。

In [ ]:
effect_summary = summarize_effects(effects, seed=SEED, n_bootstrap=N_BOOTSTRAP)
effect_summary.to_csv(OUTPUT_DIR / 'effect_summary.csv', index=False)
display(effect_summary)
show_figure(plot_effect_summary(effects, MODELS), '06_perturbation_vs_plate')

## 7. 对照中的批次近邻富集：高维定量诊断

只在同 kind、同生物 context、同控制家族的阴性对照中找余弦近邻：
- plate 指标进一步限制在同 experiment 内，比较其中不同 plate。
- experiment 指标比较相同生物/控制条件下的不同 experiment。

`富集度＝近邻中同批次比例 ÷ 随机期望比例`。随机期望为 `(该批次候选数−1)/(全部候选数−1)`，排除 query 自身。
参考线为 1：明显高于 1 支持同批次聚集；不是越接近 0 越好。
每个匹配子集每板最多抽 16 个对照、总计最多 1000 个，所有模型复用相同 wells。
相似度并列采用期望近邻计数，避免任意行顺序制造聚类；先按批次等权，再按匹配子集等权汇总。
缺两个可比批次、控制太少或该批次无其他控制的情况记录为不可评价。

In [ ]:
neighbor_results = control_neighbor_enrichment(embeddings, meta, k=NEIGHBOR_K, seed=SEED)
for name, frame in neighbor_results.items():
    frame.to_csv(OUTPUT_DIR / f'{name}.csv', index=False)
display(neighbor_results['neighbors'])
display(neighbor_results['neighbor_exclusions'])
show_figure(plot_neighbor_summary(neighbor_results['neighbors'], MODELS), '07_batch_neighbor_enrichment')

## 8. 七模型诊断汇总与可追溯输出

共同孔位相同，不意味着每项比较都存在；每项指标同时看可评价组数、对照覆盖、孔对数和匹配 context。
不把多个指标强行合成总分：好的 embedding 应同时保留跨板扰动一致性，并减少不必要的批次相关结构。
相对对照的偏离用于描述扰动强度，不设统一优劣方向。

本次仅新建诊断输出子目录，不覆盖已有 retrieval 任务成绩。明细包括共同 wells、孔对计划、排除原因、分组相似度、差值、置信区间、近邻基线以及降维坐标。

In [ ]:
overview = effect_summary.pivot(index=['model', 'kind'], columns='metric', values='mean').reindex(
    pd.MultiIndex.from_product([MODELS, sorted(meta.kind.unique())], names=['model', 'kind']))
neighbor_overview = neighbor_results['neighbors'].pivot(index=['model', 'kind'], columns='batch_level', values='enrichment').rename(
    columns={'plate': 'control_plate_neighbor_enrichment', 'experiment': 'control_experiment_neighbor_enrichment'})
overview = overview.join(neighbor_overview)
display(overview)
overview.to_csv(OUTPUT_DIR / 'seven_model_overview.csv')
run_report = {
    'scope': 'Descriptive common-well diagnostics; not held-out retrieval or causal variance attribution',
    'models': MODELS, 'model_files': {m: [str(p) for p in MODEL_FILES[m]] for m in MODELS},
    'metadata': str(METADATA_PATH), 'splits_path': str(SPLITS_PATH), 'selected_splits': SELECTED_SPLITS,
    'seed': SEED, 'metric': 'signed cosine; unit normalization only', 'control_rules': CONTROL_RULES,
    'limits': {'perturbations_per_kind': MAX_PERTURBATIONS_PER_KIND, 'wells_per_perturbation_plate': MAX_WELLS_PER_PERTURBATION_PLATE,
               'controls_per_plate': MAX_CONTROLS_PER_PLATE, 'anchors': MAX_ANCHORS_PER_PERTURBATION,
               'reference_plates': MAX_REFERENCE_PLATES, 'pairs_per_relation': MAX_PAIRS_PER_RELATION,
               'neighbor_k': NEIGHBOR_K, 'projection_points': MAX_PROJECTION_POINTS},
    'input_report': inputs['report'], 'n_pair_occurrences': len(pair_plan),
    'projection_reports': projection_reports, 'bootstrap_unit': 'gene / compound clusters',
}
(OUTPUT_DIR / 'diagnostic_protocol.json').write_text(json.dumps(run_report, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
print('Saved diagnostics:', OUTPUT_DIR)

# 附录 · 现有 val-query / train-gallery 下游任务

下面保留原 03 的 mAP/MRR、Recall、percent replicating、活性比例和配对差值。
它们直接读取已有正式 retrieval 结果，不由上面的描述性抽样重新计算。
没有正式结果时明确跳过；七模型诊断仍完整运行。

In [ ]:
TASKS = ('activity', 'same-guide', 'compound', 'cross-guide')
PRIMARY = {task: 'mrr' if task == 'cross-guide' else 'map' for task in TASKS}
models = MODELS
HAVE_RETRIEVAL_RESULTS = source_protocol.get('format') == 'phenostate-well-evaluation-v2' and (RESULT_DIR / 'summary.csv').is_file()
protocol = source_protocol
coverage = json.loads((RESULT_DIR / 'coverage.json').read_text(encoding='utf-8')) if HAVE_RETRIEVAL_RESULTS else {}
summary = pd.read_csv(RESULT_DIR / 'summary.csv') if HAVE_RETRIEVAL_RESULTS else pd.DataFrame()
results = {}
if HAVE_RETRIEVAL_RESULTS:
    for model in models:
        for task in TASKS:
            p = RESULT_DIR / model / task / 'metrics.json'
            if p.is_file():
                results[model, task] = json.loads(p.read_text(encoding='utf-8'))
else:
    print('No existing formal retrieval results; appendix panels are skipped.')

### A1. 协议、来源和覆盖率

先核对 train gallery、query split、预处理、相似度和各任务可评价率。
embedding 覆盖率与任务覆盖率的分母不同；缺失/不可评价记为 NA，不用 0 代替。
整板划分只规定当前评价孔位，不能撤销公开模型已经接触过 RxRx3 的预训练历史。

In [ ]:
if HAVE_RETRIEVAL_RESULTS:
    display(pd.Series({key: protocol.get(key) for key in ('query_split', 'gallery_split', 'tasks', 'metric',
            'preprocessing', 'seed', 'n_bootstrap', 'n_permutations', 'n_null', 'split_scope')}, name='value').to_frame())
    display(pd.Series({key: value for key, value in coverage.items()
                       if isinstance(value, (int, float))}, name='value').to_frame())
    display(pd.DataFrame([{'model': name, 'public_reference': name in PUBLIC_MODELS,
            'dimension': info.get('embedding_dimension'), 'source_kind': info.get('source_kind'),
            'provenance': info.get('checkpoint_verification')}
            for name, info in protocol['models'].items()]))
    columns = [c for c in ('model', 'task', 'score', 'n_evaluable_queries', 'n_groups', 'overall_query_coverage',
               'overall_group_coverage', 'replication_coverage', 'excluded_reasons') if c in summary]
    display(summary[columns])

    def points(ax, records, metric, bounds=(0, 1), random_field=None):
        for position, (label, stats) in enumerate(records):
            score = stats.get(metric)
            if score is None or not np.isfinite(score):
                ax.text(0.02, position, 'NA', va='center')
                continue
            interval = stats.get('ci95', {}).get(metric)
            if interval is not None and np.isfinite(interval).all():
                ax.hlines(position, interval[0], interval[1], color='tab:blue', linewidth=2)
            ax.scatter(score, position, marker='^' if label.split(' / ')[0] in PUBLIC_MODELS else 'o', color='tab:blue', clip_on=False)
            if random_field and stats.get(random_field) is not None:
                ax.scatter(stats[random_field], position, marker='x', color='gray')
        ax.set_yticks(range(len(records)), [label for label, _ in records])
        ax.set_ylim(-0.5, max(len(records) - 0.5, 0.5))
        if bounds is not None:
            ax.set_xlim(*bounds)
        ax.invert_yaxis()
        ax.set_xlabel(metric)
        ax.grid(axis='x', alpha=0.25)
        if not records:
            ax.text(0.5, 0.5, 'Not evaluated', transform=ax.transAxes, ha='center')
else:
    print('Formal retrieval panel unavailable; see the seven-model diagnostics above.')

### A2. 主指标与组级 95% CI

activity、same-guide、compound 看 mAP，cross-guide 看 MRR。四个面板分别解释，不合成为一个总分。
三角点为配置中的公开参考，圆点为其他模型；灰色叉为协议提供的随机基线。
CI 直接读取扰动组 bootstrap；它不是训练随机种子的不确定性。

In [ ]:
if HAVE_RETRIEVAL_RESULTS:
    fig, axes = plt.subplots(2, 2, figsize=(13, max(7, len(models) * 0.8)), squeeze=False)
    for ax, task in zip(axes.flat, TASKS):
        records = [(name, results[name, task]['summary']) for name in models if (name, task) in results]
        points(ax, records, PRIMARY[task], random_field='random_mrr' if task == 'cross-guide' else 'random_ap')
        ax.set_title(task)
    plt.tight_layout()
    plt.show()
else:
    print('Formal retrieval panel unavailable; see the seven-model diagnostics above.')

### A3. Recall@K、重复一致性与显著活性

Recall 是找回全部正例的比例，不是 Hit@K。percent replicating、percent active 已按 0–100 存储，不再乘 100。
显著活性按 CRISPR/COMPOUND 分开，受严格对照匹配、重复数、置换次数及 BH 阈值影响。
energy distance 另看诊断表，不能与 mAP/MRR 放在同一评分轴上。

In [ ]:
if HAVE_RETRIEVAL_RESULTS:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), squeeze=False)
    for ax, task in zip(axes.flat, ('same-guide', 'compound', 'cross-guide')):
        for name in models:
            if (name, task) not in results:
                continue
            stats = results[name, task]['summary']
            ks = protocol['ks']
            ys = [np.nan if stats.get(f'recall@{k}') is None else stats[f'recall@{k}'] for k in ks]
            ax.plot(ks, ys, 'o-', label=name)
        ax.set(title=task, xlabel='K', ylabel='Recall@K', ylim=(0, 1))
        ax.set_xticks(protocol['ks'])
        ax.grid(alpha=0.2)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(2, 2, figsize=(13, max(7, len(models) * 0.8)), squeeze=False)
    for ax, task in zip(axes[0], ('same-guide', 'compound')):
        records = [(name, results[name, task]['summary']) for name in models if (name, task) in results]
        points(ax, records, 'percent_replicating', bounds=(0, 100))
        ax.set_title(task + ' / replicating (%)')
    activity_rows = []
    for ax, kind in zip(axes[1], ('CRISPR', 'COMPOUND')):
        records = []
        for name in models:
            stats = results.get((name, 'activity'), {}).get('summary', {}).get('by_kind', {}).get(kind)
            if stats:
                records.append((name, stats))
                activity_rows.append({'model': name, 'type': kind, **{key: stats.get(key) for key in
                                     ('n_groups', 'n_tests', 'percent_active', 'percent_energy_active', 'energy_distance')}})
        points(ax, records, 'percent_active', bounds=(0, 100))
        ax.set_title(kind + ' / AP-significant activity (%)')
    plt.tight_layout()
    plt.show()
    display(pd.DataFrame(activity_rows))
else:
    print('Formal retrieval panel unavailable; see the seven-model diagnostics above.')

### A4. 各扰动组表现分布

ECDF 使用已保存的分组 AP/MRR，每个扰动组一票；不把重复 well 当成独立生物学重复。
曲线越偏右表示较多组得分较高，但应结合覆盖率与主指标 CI。

In [ ]:
if HAVE_RETRIEVAL_RESULTS:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), squeeze=False)
    for ax, task in zip(axes.flat, TASKS):
        field = 'mrr' if task == 'cross-guide' else 'ap'
        for name in models:
            path = RESULT_DIR / name / task / 'groups.csv'
            if not path.is_file():
                continue
            frame = pd.read_csv(path)
            if field not in frame:
                continue
            values = np.sort(pd.to_numeric(frame[field], errors='coerce').dropna().to_numpy())
            if len(values):
                ax.step(values, np.arange(1, len(values) + 1) / len(values), where='post', label=f'{name} (n={len(values)})')
        ax.set(title=task, xlabel=field, ylabel='Fraction of perturbation groups', xlim=(0, 1), ylim=(0, 1))
        ax.grid(alpha=0.2)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print('Formal retrieval panel unavailable; see the seven-model diagnostics above.')

### A5. 配对差值：相对于一个参考模型

只读 `paired_differences.csv` 的配对扰动 bootstrap CI，不能用两个独立 CI 的端点相减。
正值代表 other − baseline > 0。不同任务分面展示；单模型运行时无配对结果。

In [ ]:
if HAVE_RETRIEVAL_RESULTS and (RESULT_DIR / 'paired_differences.csv').is_file():
    BASELINE_MODEL = models[0]  # 可改为 'OpenPhenom' 或某个自训练模型名。
    paired = pd.read_csv(RESULT_DIR / 'paired_differences.csv')
    fig, axes = plt.subplots(2, 2, figsize=(12, max(7, len(models) * 0.7)), squeeze=False)
    for ax, task in zip(axes.flat, TASKS):
        records = []
        for row in paired.loc[paired.task.eq(task)].itertuples():
            if BASELINE_MODEL not in (row.model_a, row.model_b):
                continue
            sign = 1 if row.model_a == BASELINE_MODEL else -1
            other = row.model_b if sign == 1 else row.model_a
            lo, hi = row.ci95_lower, row.ci95_upper
            interval = [lo, hi] if sign == 1 else [-hi, -lo]
            records.append((other, {'delta': sign * row.delta_b_minus_a, 'ci95': {'delta': interval}}))
        points(ax, records, 'delta', bounds=None)
        ax.axvline(0, linestyle='--', color='gray')
        ax.set_title(task + ' / other - ' + BASELINE_MODEL)
    plt.tight_layout()
    plt.show()
else:
    print('Formal retrieval panel unavailable; see the seven-model diagnostics above.')

### A6. 选择模型、任务和扰动组查看诊断

查看每组 AP/MRR、query 排除原因、activity 匹配 block，以及 PR 的观测相似度与 null 95% 阈值。
可填写 `GROUP_LABEL` 筛选单组。表型活性不是独立的处理预测准确率；无有效对照或 null 的结果不记为阴性。

In [ ]:
if HAVE_RETRIEVAL_RESULTS and results:
    MODEL = next(name for name in models if any((name, task) in results for task in TASKS))
    TASK = next(task for task in TASKS if (MODEL, task) in results)
    GROUP_LABEL = ''
    DISPLAY_ROWS = 20
    result = results[MODEL, TASK]
    display(pd.Series(result['summary'], name='value').to_frame())
    tables = {}
    for filename in ('groups.csv', 'queries.csv', 'blocks.csv', 'replicates.csv'):
        path = RESULT_DIR / MODEL / TASK / filename
        if not path.is_file():
            continue
        frame = pd.read_csv(path)
        if GROUP_LABEL and 'perturbation_label' in frame:
            frame = frame.loc[frame.perturbation_label.eq(GROUP_LABEL)]
        tables[filename] = frame
        field = 'mrr' if TASK == 'cross-guide' else 'ap'
        if filename == 'groups.csv' and field in frame:
            frame = frame.sort_values(field)
        print(filename, '| rows:', len(frame))
        display(frame.head(DISPLAY_ROWS))
    replicates = tables.get('replicates.csv', pd.DataFrame())
    if {'median_similarity', 'null_q95'} <= set(replicates):
        valid = replicates[['median_similarity', 'null_q95']].dropna()
        if not valid.empty:
            fig, ax = plt.subplots(figsize=(5, 5))
            ax.scatter(valid.null_q95, valid.median_similarity, s=15, alpha=0.6)
            low, high = valid.to_numpy().min(), valid.to_numpy().max()
            ax.plot([low, high], [low, high], '--', color='gray')
            ax.set(xlabel='Matched null 95% threshold', ylabel='Observed replicate similarity', title=MODEL + ' / ' + TASK)
            plt.tight_layout()
            plt.show()
else:
    print('Formal retrieval panel unavailable; see the seven-model diagnostics above.')